# Notebook 06: anomalias de perfil e caracterização das faixas

**Unidade curricular:** Dados e Aprendizagem Automática (DAA), 2026/2027

Este notebook usa o modelo para duas análises. Primeiro, procura registos em que a idade declarada está longe da idade prevista, o que pode indicar erro de registo ou perfil invulgar. Segundo, descreve cada faixa etária em termos sociais e de saúde.

Trabalhamos sobre as previsões out-of-fold do melhor classificador puro, sem escalas nem stacking. Estes dois passos distorcem os priors e escondem anomalias. Cruzamos as previsões com o texto original do Notebook 01. Não há treino nesta parte.


## Carga e medida de discrepância

A discrepância é a diferença entre a classe prevista e a classe real, em número de faixas. Uma faixa corresponde a cerca de cinco anos. Definimos anomalia como uma diferença absoluta igual ou superior a 3, ou seja, 15 anos ou mais.

O erro por uma faixa é o mais comum e fica longe deste limiar. Mesmo assim, o limiar produz casos suficientes para caracterizar.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_theme(style='whitegrid')

DATA_DIR = '../data'
LIMIAR = 3

y = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'), usecols=['PID', 'AGE_RANGE'],
                index_col='PID')['AGE_RANGE']
t1 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), index_col='PID',
                 keep_default_na=False, na_values=[''])
import glob as _g
OOFF = os.path.join(DATA_DIR, 'combined_oof.npz') if os.path.exists(os.path.join(DATA_DIR, 'combined_oof.npz')) else os.path.join(DATA_DIR, 'nb04_oof.npz')
print('OOF de:', OOFF.split('/')[-1])
z = np.load(OOFF)
pred = pd.Series(z['oof'], index=y.index)
d = (pred - y).rename('gap')
ad = d.abs()
print('exata:', round(float((ad == 0).mean()), 3), '| off-by-one:', round(float((ad == 1).mean()), 3))
print('anomalias |d|>=%d:' % LIMIAR, int((ad >= LIMIAR).sum()),
      '| >=4:', int((ad >= 4).sum()))

plt.figure(figsize=(9, 4))
sns.countplot(x=ad.clip(upper=6), hue=ad.clip(upper=6), legend=False, palette='viridis')
plt.title('Distribuicao de |prevista - real| (6+ agrupado)')
plt.xlabel('|gap| em classes')
plt.tight_layout()
plt.show()

## Direção do erro e controlo de routing

Uma diferença positiva significa que o perfil foi previsto como mais velho do que a idade real. Uma diferença negativa significa que foi previsto como mais novo. Antes de chamar anomalia biológica a um caso, verificamos o routing. Um reformado previsto como 15-17, ou um estudante com 75-79, sugere erro de registo e não biologia.

Comparamos os dois sentidos com a população de referência em situação profissional, sexo, número de comorbilidades e carga de routing.


In [ ]:
cd = [c for c in t1.columns if c.startswith('CD1')]
base = pd.DataFrame({'N_CD_SIM': (t1[cd] == 'Sim').sum(axis=1)})
base['N_NA'] = (t1.select_dtypes(include='str') == 'Não aplicável').sum(axis=1)
base['gap'] = d
base['grupo'] = np.where(ad >= LIMIAR, np.where(d > 0, 'envelhecida', 'rejuvenescida'), 'normal')
print(base['grupo'].value_counts().to_string())
print()
print(base.groupby('grupo')[['N_CD_SIM', 'N_NA']].mean().round(2).to_string())
print()
print('MAINSTAT das envelhecidas:')
print(t1.loc[base['grupo'] == 'envelhecida', 'MAINSTAT'].value_counts().head(6).to_string())
print('MAINSTAT das rejuvenescidas:')
print(t1.loc[base['grupo'] == 'rejuvenescida', 'MAINSTAT'].value_counts().head(6).to_string())

plt.figure(figsize=(10, 4))
sns.barplot(data=base.reset_index(), x='N_CD_SIM', y='grupo', errorbar='sd', orient='y')
plt.title('N comorbilidades por grupo')
plt.tight_layout()
plt.show()

## Casos para o relatório

Mostramos os maiores desvios nos dois sentidos, com um perfil compacto: situação profissional, estado civil, escolaridade, saúde auto-percebida, número de comorbilidades e lista das doenças declaradas. Estes casos alimentam a discussão entre erro do modelo e perfil invulgar.


In [ ]:
COLS_CASO = ['MAINSTAT', 'MARSTALEGAL', 'HATLEVEL', 'HS1', 'SEX']
top = d.loc[ad >= LIMIAR].sort_values(key=abs, ascending=False).head(6)
for pid, gap in top.items():
    print('--- PID', pid, '| real', int(y.loc[pid]), '| prevista', int(pred.loc[pid]), '| gap', int(gap))
    print('   ', ' | '.join(str(t1.loc[pid, c])[:42] for c in COLS_CASO))
    print('    N_CD_SIM:', int(base.loc[pid, 'N_CD_SIM']), '| doencas:',
          [c for c in cd if t1.loc[pid, c] == 'Sim'][:8] or 'nenhuma')
    print()

## Retrato das 16 faixas

Esta secção descreve cada faixa: dimensão, percentagem de mulheres, de empregados, de reformados e de viúvos, comorbilidades médias e carga de routing. É a tabela de base para a caracterização no relatório. O mapa de calor mostra as transições de ciclo de vida que o modelo aprendeu, como estudo, emprego e reforma, e solteiro, casado e viúvo.


In [ ]:
g = y.to_frame().assign(Mulher=(t1['SEX'] == 'Mulher').values,
                     Empregada=(t1['MAINSTAT'].str.contains('Tem um emprego').fillna(False)).values,
                     Reformada=(t1['MAINSTAT'].str.contains('Reformado').fillna(False)).values,
                     Viuva=(t1['MARSTALEGAL'] == 'Viúvo').values,
                     N_CD_SIM=base['N_CD_SIM'].values,
                     N_NA=base['N_NA'].values).groupby('AGE_RANGE')
retrato = pd.DataFrame({'n': g.size(), '%Mulher': g['Mulher'].mean() * 100,
                        '%Emprego': g['Empregada'].mean() * 100, '%Reforma': g['Reformada'].mean() * 100,
                        '%Viuvo': g['Viuva'].mean() * 100, 'N_CD_medio': g['N_CD_SIM'].mean(),
                        'N_NA_medio': g['N_NA'].mean()}).round(1)
print(retrato.to_string())

plt.figure(figsize=(12, 5))
sns.heatmap(retrato[['%Mulher', '%Emprego', '%Reforma', '%Viuvo']].T, annot=True, fmt='.0f',
            cmap='viridis')
plt.title('Transicoes de ciclo de vida por faixa etaria (%)')
plt.tight_layout()
plt.show()

## Grupos não supervisionados (KMeans, para o relatório)

KMeans k=16 nas 339 features standardizadas (sem alvo): os clusters recuperam as
faixas etárias? Um ARI baixo mostra que o sinal de idade está difuso em combinações,
o que sustenta a tese do teto. Cruzamos ainda os clusters com as anomalias.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

Xnb = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'), index_col='PID').drop(columns=['AGE_RANGE'])
Xs = StandardScaler().fit_transform(Xnb.values)
km = KMeans(n_clusters=16, n_init=10, random_state=42).fit(Xs)
cl = pd.Series(km.labels_, index=Xnb.index, name='cluster')
print('ARI (clusters vs faixas):', round(float(adjusted_rand_score(y, cl)), 4))

perf = pd.DataFrame({'n': cl.value_counts().sort_index(),
                     'idade_media': y.groupby(cl).mean(),
                     'top1': y.groupby(cl).agg(lambda s: s.value_counts().index[0]),
                     '%top1': (y.groupby(cl).apply(lambda s: (s == s.value_counts().index[0]).mean()) * 100),
                     '%anomalia': (ad >= LIMIAR).groupby(cl).mean().mul(100)}).round(2)
print(perf.sort_values('idade_media').to_string())

ct = pd.crosstab(cl, y, normalize='index') * 100
plt.figure(figsize=(14, 7))
sns.heatmap(ct, annot=False, cmap='viridis')
plt.title('Cluster (linhas) x faixa real (colunas), % do cluster')
plt.xlabel('AGE_RANGE')
plt.ylabel('cluster')
plt.tight_layout()
plt.show()


## Conclusões

- O modelo acerta 47.6% dos casos e erra por uma faixa em 39.9%. Há 260 anomalias com diferença absoluta igual ou superior a 3, e 53 delas têm diferença de 4 ou mais. O melhor modelo produz menos anomalias do que o modelo fraco anterior, que tinha 466. Os desvios maiores vêm do modelo, não dos dados.

- As anomalias têm menos comorbilidades do que a população. Envelhecidas: 2.05. Rejuvenescidas: 1.75. População: 2.58. Das 141 rejuvenescidas, 55 são reformados (39%). Os casos típicos são os PID 7191 e 11707, entre 50 e 54 anos, estudantes, previstos como 20-24; o 8457, entre 50 e 54, solteira, prevista como 20-24; o 10646, entre 80 e 84, com quatro doenças, previsto como 50-54; e o 1265, com 85 ou mais e zero doenças.

- O retrato por faixa não muda, porque vem dos rótulos reais e não do modelo. A tabela serve diretamente para o relatório.

- O KMeans (k=16, sem alvo) não recupera as faixas (ARI 0.11). Os clusters ordenam-se por idade média mas com pureza fraca (14-48%). Isto sustenta a tese do teto: o sinal de idade está difuso em combinações. O cluster 13, o mais velho, concentra 4.27% de anomalias.
